In [2]:
import pandas as pd

url = "https://raw.githubusercontent.com/biplav-s/course-tai/main/sample-code/common-data/water/WaterAtlas-ManySites.csv"

df = pd.read_csv(url, engine="python", on_bad_lines="skip")
df.shape, df.head()


((64188, 17),
                                   DataSourceName  DataSourceCode  \
 0  LAKEWATCH Supplemental Water Quality Sampling  LAKEWATCH_SUPP   
 1  LAKEWATCH Supplemental Water Quality Sampling  LAKEWATCH_SUPP   
 2  LAKEWATCH Supplemental Water Quality Sampling  LAKEWATCH_SUPP   
 3  LAKEWATCH Supplemental Water Quality Sampling  LAKEWATCH_SUPP   
 4  LAKEWATCH Supplemental Water Quality Sampling  LAKEWATCH_SUPP   
 
            StationID ActualStationID  Latitude_DD  Longitude_DD  \
 0  Bugg Springs-Lake             NaN     28.75361     -81.90444   
 1  Bugg Springs-Lake             NaN     28.75361     -81.90444   
 2  Bugg Springs-Lake             NaN     28.75361     -81.90444   
 3  Bugg Springs-Lake             NaN     28.75361     -81.90444   
 4  Bugg Springs-Lake             NaN     28.75361     -81.90444   
 
                 SampleDate SampleTime  ActivityDepth ActivityDepthUnit  \
 0  1991-08-18 00:00:00.000   00:00:00            NaN               NaN   
 1  1991-0

In [3]:
df.columns


Index(['DataSourceName', 'DataSourceCode', 'StationID', 'ActualStationID',
       'Latitude_DD', 'Longitude_DD', 'SampleDate', 'SampleTime',
       'ActivityDepth', 'ActivityDepthUnit', 'Characteristic', 'ResultValue',
       'ResultUnit', 'ValueQualifier', 'ResultComment', 'WaterbodyID',
       'WaterbodyName'],
      dtype='object')

In [7]:
ph_df = df[df["Characteristic"] == "pH"].copy()

ph_df.shape

ph_df["ResultValue"] = pd.to_numeric(ph_df["ResultValue"], errors="coerce")
ph_df = ph_df.dropna(subset=["ResultValue"])


In [8]:
ph_df["SAFE-PH"] = np.where(
    (ph_df["ResultValue"] > 6.5) & (ph_df["ResultValue"] <= 8.5),
    "yes",
    "no"
)

ph_df["SAFE-PH"].value_counts()


,count
SAFE-PH,
yes,383
no,89


In [12]:
from sklearn.model_selection import train_test_split

X = ph_df[["Latitude_DD", "Longitude_DD"]]
y = ph_df["SAFE-PH"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

X_train.shape, X_test.shape


((377, 2), (95, 2))

In [13]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, precision_score, recall_score, f1_score

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

scoring = {
    "accuracy": "accuracy",
    "precision_yes": make_scorer(precision_score, pos_label="yes"),
    "recall_yes": make_scorer(recall_score, pos_label="yes"),
    "f1_yes": make_scorer(f1_score, pos_label="yes")
}


In [14]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

dt = DecisionTreeClassifier(random_state=42)

# 10-fold CV
cv_results_dt = cross_validate(dt, X_train, y_train, cv=cv, scoring=scoring)

print("Decision Tree — 10-fold CV")
print("Accuracy      :", cv_results_dt["test_accuracy"].mean())
print("Precision(yes):", cv_results_dt["test_precision_yes"].mean())
print("Recall(yes)   :", cv_results_dt["test_recall_yes"].mean())
print("F1(yes)       :", cv_results_dt["test_f1_yes"].mean())

# Test evaluation (20%)
dt.fit(X_train, y_train)
pred_dt = dt.predict(X_test)

print("\nDecision Tree — TEST (20%)")
print("Accuracy:", accuracy_score(y_test, pred_dt))
print("Confusion Matrix:\n", confusion_matrix(y_test, pred_dt))
print(classification_report(y_test, pred_dt))


Decision Tree — 10-fold CV
Accuracy      : 0.8567567567567567
Precision(yes): 0.9167692755324847
Recall(yes)   : 0.9088172043010753
F1(yes)       : 0.9114378396243081

Decision Tree — TEST (20%)
Accuracy: 0.8421052631578947
Confusion Matrix:
 [[14  4]
 [11 66]]
              precision    recall  f1-score   support

          no       0.56      0.78      0.65        18
         yes       0.94      0.86      0.90        77

    accuracy                           0.84        95
   macro avg       0.75      0.82      0.77        95
weighted avg       0.87      0.84      0.85        95



In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

lr = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=2000))
])

# 10-fold CV
cv_results_lr = cross_validate(lr, X_train, y_train, cv=cv, scoring=scoring)

print("Logistic Regression — 10-fold CV")
print("Accuracy      :", cv_results_lr["test_accuracy"].mean())
print("Precision(yes):", cv_results_lr["test_precision_yes"].mean())
print("Recall(yes)   :", cv_results_lr["test_recall_yes"].mean())
print("F1(yes)       :", cv_results_lr["test_f1_yes"].mean())

# Test evaluation (20%)
lr.fit(X_train, y_train)
pred_lr = lr.predict(X_test)

print("\nLogistic Regression — TEST (20%)")
print("Accuracy:", accuracy_score(y_test, pred_lr))
print("Confusion Matrix:\n", confusion_matrix(y_test, pred_lr))
print(classification_report(y_test, pred_lr))


Logistic Regression — 10-fold CV
Accuracy      : 0.8460881934566146
Precision(yes): 0.8637953031335384
Recall(yes)   : 0.9643010752688171
F1(yes)       : 0.9107125939736782

Logistic Regression — TEST (20%)
Accuracy: 0.8105263157894737
Confusion Matrix:
 [[ 7 11]
 [ 7 70]]
              precision    recall  f1-score   support

          no       0.50      0.39      0.44        18
         yes       0.86      0.91      0.89        77

    accuracy                           0.81        95
   macro avg       0.68      0.65      0.66        95
weighted avg       0.80      0.81      0.80        95



In [16]:
location_summary = ph_df.groupby(
    ["WaterbodyName", "Latitude_DD", "Longitude_DD"]
)["SAFE-PH"].apply(lambda x: (x == "no").sum()).reset_index(name="UnsafeCount")

location_summary = location_summary.sort_values("UnsafeCount", ascending=False)

print("Top 10 Most Unsafe Locations:")
display(location_summary.head(10))

print("\nBottom 10 Least Unsafe Locations:")
display(location_summary.tail(10))


Top 10 Most Unsafe Locations:


,WaterbodyName,Latitude_DD,Longitude_DD,UnsafeCount
23,Palatlakaha River,28.748033,-81.874853,25
7,Church Lake,28.642111,-81.840597,13
13,Palatlakaha River,28.679167,-81.884778,12
8,Church Lake,28.645000,-81.846400,11
20,Palatlakaha River,28.744200,-81.872800,11
11,Kess Lake,28.664713,-81.842010,2
27,Palatlakaha River,28.754084,-81.875130,2
29,Turkey Lake,28.701280,-81.850390,2
26,Palatlakaha River,28.753987,-81.874909,2
14,Palatlakaha River,28.679456,-81.884452,1



Bottom 10 Least Unsafe Locations:


,WaterbodyName,Latitude_DD,Longitude_DD,UnsafeCount
3,Bugg Spring,28.753610,-81.904440,0
0,Bugg Spring,28.751943,-81.901664,0
6,Bugg Spring Run,28.754100,-81.900602,0
2,Bugg Spring,28.751987,-81.901517,0
21,Palatlakaha River,28.747222,-81.874861,0
18,Palatlakaha River,28.737240,-81.871092,0
25,Palatlakaha River,28.750833,-81.875833,0
24,Palatlakaha River,28.750727,-81.875903,0
22,Palatlakaha River,28.747893,-81.874693,0
28,Turkey Lake,28.698290,-81.852078,0
